# Predicting Airline Satisfaction

In [ ]:
# import required libbraries
import time

import numpy as  np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)


print("Libraries imported successfully...Bliss")

In [ ]:
sample_submission = "/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv"
test = "/kaggle/input/competitions/playground-series-s6e10/test.csv"
train = "/kaggle/input/competitions/playground-series-s6e10/train.csv"

df_sample = pd.read_csv(sample_submission)
df_test = pd.read_csv(test)
df_train = pd.read_csv(train)

print("Dataset loaded successfully")

In [ ]:
df_sample.head()

In [ ]:
df_test.head()

In [ ]:
df_train.head()

In [ ]:
print(f"sample_submission Data Shape:")
print(f"Rows: {df_sample.shape[0]} | Columns: {df_sample.shape[1]}\n")

print(f"Test Data Shape:")
print(f"Rows: {df_test.shape[0]} | Columns: {df_test.shape[1]}\n")

print(f"Train Data Shape:")
print(f"Rows: {df_train.shape[0]} | Columns: {df_train.shape[1]}")

In [ ]:
df_sample_list = df_sample.columns.tolist()
df_train_list = df_train.columns.tolist()
df_test_list = df_test.columns.tolist()

print("df_sample_list\n", df_sample_list, "\n")
print("df_train_list\n", df_train_list, "\n")
print("df_test_list\n", df_test_list)

In [ ]:
print(df_train["satisfaction"].value_counts())
print(df_train["satisfaction"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
## Missing values

dfs = {
    "Train" : df_train,
    "Test" : df_test
}

for name, df in dfs.items():
    print(f"Missing values in {name} Data")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print("-"*50)

In [ ]:
sns.countplot(data=df_train, x="satisfaction")
plt.title("Airline satisfaction Distribution")
plt.show()

In [ ]:
cat_cols = ["Gender", "Class", "Type of Travel", "Customer Type"]

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
axes = axes.flatten()  # Flatten to easily iterate through axes with a single loop

for i, col in enumerate(cat_cols):
    
    # Plot on the specific subplot axis
    sns.barplot(data=df_train, x=col, y="satisfaction", errorbar=None, ax=axes[i])
    axes[i].set_title(f"Satisfaction Rate By \"{col}\"")
    axes[i].set_ylabel("Satisfaction Rate")
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Analyze airline service rating

In [ ]:
print("df_train_list\n", df_train_list, "\n")

In [ ]:
service_feature = ["service", "comfort", "food", "drink", "cleanliness", "entertainment", "boarding", "wifi", "check-in", "checkin", "online booking", "leg room"]

ser_fest = []
print("Service-related columns:")
for col in df_train.columns:
    # check if any keyword in service_feature is a substring of the column name
    if any(feat in col.lower() for feat in service_feature):
        ser_fest.append(col)
        print(f"-{col}")

In [ ]:
print(ser_fest)

In [ ]:
comparison = df_train.groupby("satisfaction")[ser_fest].mean().T
print(comparison.round(2))

In [ ]:
#Visualize the difference

comparison.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.title("Average service Ratings by Satisfaction")
plt.ylabel("Avg Rating")
plt.xlabel("Service Feature")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Analyze flight delayes

In [ ]:
delay_feature = ["Departure Delay in Minutes",
                  "Arrival Delay in Minutes"
                  ]

delay_fest = []
print("Delay-related columns:")
for col in df_train.columns:
    # check if any keyword in delay_feature is a substring of the column name
    if any(feat.lower() in col.lower() for feat in delay_feature):
        delay_fest.append(col)
        print(f"-{col}")

In [ ]:
# Visualize the distribution

for col in delay_fest:
    plt.figure(figsize=(8, 4))

    sns.boxplot(
        data=df_train,
        x="satisfaction",
        y=col,
        showfliers=False
    )

    plt.title(f"\"{col}\" by Satisfaction")
    plt.tight_layout()
    plt.show()

## Analyze numerical feature

In [ ]:
num_cols = df_train.select_dtypes(include="number").columns.tolist()

for col in num_cols:
    if col == "id":
        num_cols.remove(col)
print(num_cols)

In [ ]:
# summary statistics
display(df_train[num_cols].describe().drop("count").T.round(2))

In [ ]:
# plot bumerical distribution

features_to_plot = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]

for col in features_to_plot:
    plt.figure(figsize=(7, 4))

    sns.histplot(
        data=df_train,
        x=col,
        bins=40,
        kde=False
    )

    plt.title(f"Distribution of \"{col}\"")
    plt.tight_layout()
    plt.show()

## Correlation analysis

In [ ]:
# Include the Boolean target as a numeric variable
corr_data = df_train[num_cols].copy()
corr_data["satisfaction"] = (
    df_train["satisfaction"].astype(int)
)

correlation = corr_data.corr(numeric_only=True)

# show correlation with the target
target_corr = (
    correlation["satisfaction"].drop("satisfaction").sort_values(ascending=False)
)

display(target_corr.to_frame("Correlation with satisfaction").round(3))

In [ ]:
top_features = target_corr.abs().nlargest(15).index
print(top_features)

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_data[top_features.tolist() + ["satisfaction"]].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap: Selected Features")
plt.tight_layout()
plt.show()

## EDA Conclusion

In [ ]:
eda_summary = {
    "training_rows": df_train.shape[0],
    "traing_colums": df_train.shape[1],
    "test_rows": df_test.shape[0],
    "missing_train_values": int(df_train.isnull().sum().sum()),
    "duplication_train_rows": int(df_train.isnull().sum().sum()),
    "satisfied_percentage": round(df_train["satisfaction"].mean()*100, 2),
    "not_satisfaction_percentage": round(1-df_train["satisfaction"].mean()*100,2)
}

for key, value in eda_summary.items():
    print(f"{key}: {value}")

## Data Preprocessing

### Seperate features and target

In [ ]:
X = df_train.drop(columns=["satisfaction", "id"])
y = df_train["satisfaction"].astype(int)

# saving test IDs for final submission
test_ids = df_test["id"].copy()
X_df_test = df_test.drop(columns = "id")

# now identify the columns types
cat_features = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_features = X.select_dtypes(include=["number"]).columns.tolist()

print(f"X sahpe: {X.shape}")
print(f"y sahpe: {y.shape}")
print(f"No. of categorical calumns: {len(cat_features)}")
print(f"No. of numerical calumns: {len(num_features)}")
print("-"*70)
print(f"categorical calumns:\n{cat_features}")
print("-"*110)
print(f"numerical calumns:\n{num_features}")

## Splitting data and building pipeline
*We train and test different types of ML models for better predictions*

### Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
                                                   )

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Numerical preprocessing
category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

#combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", category_pipeline, cat_features)
])

# complete model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=5000))
])

print("Preprocessing and model pipeline ready!")

In [ ]:
# Train the model
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"F1 binary: {f1_score(y_test, y_pred)}")
print(f"F1 macro: {f1_score(y_test, y_pred, average="macro")}")
print(f"F1 weighted: {f1_score(y_test, y_pred, average="weighted")}")
print("-"*80)
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("-"*70)
print(classification_report(y_test, y_pred))

### Random Forest Classifier

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators = 50,
        max_depth=14,
        min_samples_leaf=3,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ))
])

print("Random forest training started...")
start = time.time()

rf_model.fit(X_train, y_train)

training_time = time.time() - start
print("Training completed")

In [ ]:
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

print("\n======= Random Forest Result============")

print("Accuracy: ", round(rf_accuracy, 5))
print("F1 Score: ", round(rf_f1, 5))
print("ROC-AUC: ", round(rf_auc, 5))
print("Training Time: ", round(training_time, 1), "seconds")

print("===========================================")

## Gradient Boosting Classifier

In [ ]:
gbc_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=3,
    min_samples_leaf=20,
    subsample=0.8,
    random_state=42
))
])


print("Gradient Boosting training started...")
start = time.time()

gbc_model.fit(X_train, y_train)

training_time = time.time() - start
print("Training completed")

In [ ]:
gbc_pred = gbc_model.predict(X_test)
gbc_proba = gbc_model.predict_proba(X_test)[:, 1]

gbc_accuracy = accuracy_score(y_test, gbc_pred)
gbc_f1 = f1_score(y_test, gbc_pred)
gbc_auc = roc_auc_score(y_test, gbc_proba)

print("\n======= Gradient Boosting Result============")

print("Accuracy: ", round(gbc_accuracy, 5))
print("F1 Score: ", round(gbc_f1, 5))
print("ROC-AUC: ", round(gbc_auc, 5))
print("Training Time: ", round(training_time, 1), "seconds")

print("===========================================")

## Gausian Naive Bayes'

In [ ]:
gnb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB()
    )
])


print("GausianNB training started...")
start = time.time()

gnb_model.fit(X_train, y_train)

training_time = time.time() - start
print("Training completed")

In [ ]:
gnb_pred = gnb_model.predict(X_test)
gnb_proba = gnb_model.predict_proba(X_test)[:, 1]

gnb_accuracy = accuracy_score(y_test, gnb_pred)
gnb_f1 = f1_score(y_test, gnb_pred)
gnb_auc = roc_auc_score(y_test, gnb_proba)

print("\n======= Gaussian Naive Bayes' Result============")

print("Accuracy: ", round(gnb_accuracy, 5))
print("F1 Score: ", round(gnb_f1, 5))
print("ROC-AUC: ", round(gnb_auc, 5))
print("Training Time: ", round(training_time, 1), "seconds")

print("===========================================")

## CatBoost Classifier

In [ ]:
cbc_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        verbose=0
    )
    )
])


print("CatBoostClassifier training started...")
start = time.time()

cbc_model.fit(X_train, y_train)

training_time = time.time() - start
print("Training completed")

In [ ]:
cbc_pred = cbc_model.predict(X_test)
cbc_proba = cbc_model.predict_proba(X_test)[:, 1]

cbc_accuracy = accuracy_score(y_test, cbc_pred)
cbc_f1 = f1_score(y_test, cbc_pred)
cbc_auc = roc_auc_score(y_test, cbc_proba)

print("\n======= CatBoost Result============")

print("Accuracy: ", round(cbc_accuracy, 5))
print("F1 Score: ", round(cbc_f1, 5))
print("ROC-AUC: ", round(cbc_auc, 5))
print("Training Time: ", round(training_time, 1), "seconds")

print("===========================================")

## The Best model

In [ ]:
# Dictionary to store ROU_AUC values to pick best one
auc_score = {
    "Random Forest": rf_auc,
    "Gradient Boosting": gbc_auc,
    "Gausian Naive Bayes": gnb_auc,
    "CatBoost": cbc_auc
}

best_model = max(auc_score, key=auc_score.get)
print(f"The best model is \"{best_model}\" with a ROC-AUC score of \"{round(auc_score[best_model], 5)}\"")

## Final Submission & Conclusion

In [ ]:
# Train full training data
X = df_train.drop(columns=["satisfaction", "id"])
y = df_train["satisfaction"].astype(int)

# prepare the kaggle test data
X_test = df_test.drop(columns=["id"])
test_ids = df_test["id"]

# Train Random Forest on all training rows
rf_model.fit(X, y)

# predict satisfaction probabilities
test_prob = rf_model.predict_proba(X_test)[:, 1]

# create submission file
submission = pd.DataFrame({
    "id": test_ids,
    "satisfaction": test_prob
})

submission.to_csv("submission.csv", index=False)

print("submission file created successfully")

In [ ]:
print(f"Submission sahpe: {submission.shape}")
print(submission.head())
print(f"Missing values: {submission.isnull().sum().sum()}")
print(f"Probability range: {submission["satisfaction"].min()} to {submission["satisfaction"].max()}")